# RockNet v2 — Training Notebook
Two-stage multi-head rock classification: primary family head + conditional feature heads.

**Workflow:**
1. Mount Drive + clone repo
2. Load  → split → dataloaders
3. Warm-up primary head (epochs 0–4, secondary loss off)
4. Joint training (epochs 5+, full loss)
5. Temperature calibration on val set
6. Save checkpoint to Drive


## 0. Environment check

In [ ]:
import torch, sys
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Paths
Set  to wherever your capstone repo lives on Drive.

In [ ]:
from pathlib import Path

# ── Edit these two lines ───────────────────────────────────────────────
REPO_PATH     = Path("/content/drive/MyDrive/capstone")          # repo root
MANIFEST_PATH = REPO_PATH / "ML-classifications/metadata/dataset_manifest.json"
WEIGHTS_OUT   = REPO_PATH / "ML-classifications/models/best_rocknet_v2.pt"
# ────────────────────────────────────────────────────────────────────────

assert MANIFEST_PATH.exists(), f"Manifest not found: {MANIFEST_PATH}"
print("Manifest:", MANIFEST_PATH)


## 3. Install dependencies

In [ ]:
!pip install timm --quiet
import sys
sys.path.insert(0, str(REPO_PATH / "ML-classifications/scripts"))
from rocknet_v2 import (
    RockNetV2, FEATURE_SCHEMAS, LABELING_SCHEMAS, FAMILY_NAMES, FAMILY_TO_IDX,
    ROCK_FAMILIES, IGNORE_INDEX, SECONDARY_WARMUP_EPOCHS, LAMBDA_SECONDARY,
    label_to_idx, get_feature_value, get_feature_weight,
    compute_loss, save_checkpoint, load_checkpoint, train_tfms, val_tfms,
)
print("rocknet_v2 imported OK")

## 4. Training configuration

In [ ]:
CFG = {
    # Data
    "val_frac":          0.15,    # fraction of data for validation
    "test_frac":         0.15,    # fraction held out for final eval
    "min_reviewed":      True,    # if True, only use reviewed=True entries
    "use_auto_labeled":  True,    # also include auto_labeled entries (VLM-labeled, not human-confirmed)
    "source_weight":     True,    # weight sampler by entry.source_weight

    # Training
    "epochs":            60,
    "batch_size":        32,
    "lr":                3e-4,
    "weight_decay":      1e-4,
    "label_smoothing":   0.1,
    "grad_clip":         1.0,

    # Hard negative "other" class
    # Set to a folder of non-rock images, or None to skip
    "other_image_dir":   None,
    "other_max":         300,     # cap so "other" stays <=20% of training data

    # Checkpoint
    "patience":          10,      # early stopping on val primary accuracy
    "save_every":        5,       # also save a checkpoint every N epochs
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Config:", CFG)

## 5. Dataset class

In [ ]:
import json, random
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class RockDataset(Dataset):
    """
    Reads entries from dataset_manifest.json.

    Each item returns:
      (image_tensor, family_idx, feature_targets_dict, feature_weights_dict)

    feature_targets_dict: family → feat → scalar int64
      IGNORE_INDEX for entries from a different family or with n/a labels.

    feature_weights_dict: family → feat → scalar float32 in [0.5, 1.0]
      Derived from the labeler confidence rating stored in the manifest.
      1=uncertain→0.5, 2=likely→0.75, 3=certain→1.0.
      Old string-format entries and wrong-family slots default to 1.0.
    """
    def __init__(self, entries: list[dict], transform, include_other_dir=None, other_max: int = 300):
        self.entries   = list(entries)
        self.transform = transform

        # Add hard-negative "other" class from a folder of non-rock images
        if include_other_dir and Path(include_other_dir).exists():
            exts = {".jpg", ".jpeg", ".png", ".webp"}
            other_paths = sorted([
                p for p in Path(include_other_dir).rglob("*") if p.suffix.lower() in exts
            ])
            random.shuffle(other_paths)
            other_paths = other_paths[:other_max]
            for p in other_paths:
                self.entries.append({"_other_path": str(p), "family": "other", "features": {}})

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]

        # Image path: processed first, then raw, then _other_path
        img_path = (
            entry.get("_other_path")
            or entry.get("processed_path")
            or entry.get("raw_path")
        )
        img = Image.open(img_path).convert("RGB")
        tensor = self.transform(img)

        family     = entry["family"]
        family_idx = FAMILY_TO_IDX.get(family, FAMILY_TO_IDX["other"])
        raw_feats  = entry.get("features", {})

        feat_targets = {}
        feat_weights = {}
        for fam in ROCK_FAMILIES:
            feat_targets[fam] = {}
            feat_weights[fam] = {}
            for feat, classes in LABELING_SCHEMAS[fam].items():
                if family != fam:
                    feat_targets[fam][feat] = torch.tensor(IGNORE_INDEX, dtype=torch.long)
                    feat_weights[fam][feat] = torch.tensor(1.0, dtype=torch.float32)
                else:
                    feat_entry = raw_feats.get(feat, "n/a")
                    feat_targets[fam][feat] = torch.tensor(
                        label_to_idx(feat, feat_entry, classes), dtype=torch.long
                    )
                    feat_weights[fam][feat] = torch.tensor(
                        get_feature_weight(feat_entry), dtype=torch.float32
                    )

        return tensor, torch.tensor(family_idx, dtype=torch.long), feat_targets, feat_weights


def collate_feat_targets(batch):
    """Custom collate: stack image tensors + family labels; concat feature target and weight dicts."""
    imgs, family_tgts, feat_tgts_list, feat_wts_list = zip(*batch)
    imgs        = torch.stack(imgs)
    family_tgts = torch.stack(family_tgts)
    feat_tgts = {
        fam: {
            feat: torch.stack([ft[fam][feat] for ft in feat_tgts_list])
            for feat in LABELING_SCHEMAS[fam]
        }
        for fam in ROCK_FAMILIES
    }
    feat_weights = {
        fam: {
            feat: torch.stack([fw[fam][feat] for fw in feat_wts_list])
            for feat in LABELING_SCHEMAS[fam]
        }
        for fam in ROCK_FAMILIES
    }
    return imgs, family_tgts, feat_tgts, feat_weights

print("Dataset class defined")

## 6. Load manifest and split
> Splits are stratified by family AND source to prevent leakage.
> Lunar sources (lpi, astromaterials) are kept proportional across all splits.


In [ ]:
import json, random
from collections import defaultdict

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

all_entries = manifest["images"]

# Build set of usable label statuses.
# reviewed=True entries are always included.
# auto_labeled entries (VLM-labeled, not human-confirmed) are included when
# use_auto_labeled=True — their conf scores flow through to loss weights, so
# lower-confidence VLM predictions contribute less to training.
# auto_hint / unlabeled / auto_label_failed entries are excluded — they have
# too many n/a features to be useful.
usable_statuses = {"reviewed"}
if CFG.get("use_auto_labeled", True):
    usable_statuses.add("auto_labeled")

all_entries = [
    e for e in all_entries
    if e.get("label_status") in usable_statuses
    and e.get("quality") != "rejected"
    and e.get("family") in ["basalt", "anorthosite", "breccia"]
]

print(f"Total usable entries: {len(all_entries)}")
from collections import Counter
print("By family:",  dict(Counter(e["family"] for e in all_entries)))
print("By status:",  dict(Counter(e.get("label_status") for e in all_entries)))
print("By source:",  dict(Counter(e["source"] for e in all_entries)))

# Stratified split: group by (family, source), then split each group
def stratified_split(entries, val_frac, test_frac, seed=42):
    rng = random.Random(seed)
    groups = defaultdict(list)
    for e in entries:
        groups[(e["family"], e.get("source", "unknown"))].append(e)

    train, val, test = [], [], []
    for key, group in groups.items():
        rng.shuffle(group)
        n    = len(group)
        n_te = max(1, int(n * test_frac)) if n >= 3 else 0
        n_va = max(1, int(n * val_frac))  if n >= 3 else 0
        test  += group[:n_te]
        val   += group[n_te:n_te + n_va]
        train += group[n_te + n_va:]

    return train, val, test

train_entries, val_entries, test_entries = stratified_split(
    all_entries, CFG["val_frac"], CFG["test_frac"]
)
print(f"\nSplit -- train: {len(train_entries)}  val: {len(val_entries)}  test: {len(test_entries)}")

## 7. Dataloaders

In [ ]:
train_ds = RockDataset(
    train_entries, train_tfms,
    include_other_dir=CFG["other_image_dir"],
    other_max=CFG["other_max"],
)
val_ds  = RockDataset(val_entries,  val_tfms)
test_ds = RockDataset(test_entries, val_tfms)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

# Weighted sampler: oversample by source_weight (boosts lunar sources)
def make_sampler(ds):
    weights = []
    for e in ds.entries:
        w = e.get("source_weight", 1.0)
        weights.append(w)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"],
    sampler=make_sampler(train_ds) if CFG["source_weight"] else None,
    shuffle=(not CFG["source_weight"]),
    num_workers=2, pin_memory=True, collate_fn=collate_feat_targets,
)
val_loader = DataLoader(
    val_ds, batch_size=CFG["batch_size"],
    shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_feat_targets,
)
print("Dataloaders ready")


## 8. Model

In [ ]:
model = RockNetV2().to(DEVICE)

# Count trainable parameters
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total/1e6:.1f}M  Trainable: {trainable/1e6:.1f}M")


## 9. Optimizer + LR scheduler

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Lower LR for pretrained backbone, full LR for new heads
backbone_params = list(model.backbone.parameters())
head_params     = [
    p for n, p in model.named_parameters()
    if not n.startswith("backbone") and p.requires_grad
]

optimizer = AdamW([
    {"params": backbone_params, "lr": CFG["lr"] * 0.1},   # 10x lower for backbone
    {"params": head_params,     "lr": CFG["lr"]},
], weight_decay=CFG["weight_decay"])

scheduler = CosineAnnealingLR(optimizer, T_max=CFG["epochs"], eta_min=1e-6)
print("Optimizer and scheduler ready")


## 10. Evaluation helper

In [ ]:
def evaluate(model, loader, epoch=999):
    model.eval()
    total_loss = 0.0
    correct_primary = 0
    total_samples   = 0

    from collections import defaultdict
    feat_correct = defaultdict(int)
    feat_total   = defaultdict(int)

    with torch.no_grad():
        for imgs, fam_tgts, feat_tgts, feat_wts in loader:
            imgs      = imgs.to(DEVICE)
            fam_tgts  = fam_tgts.to(DEVICE)
            feat_tgts = {fam: {f: t.to(DEVICE) for f, t in v.items()} for fam, v in feat_tgts.items()}
            feat_wts  = {fam: {f: w.to(DEVICE) for f, w in v.items()} for fam, v in feat_wts.items()}

            outputs = model(imgs)
            loss, _ = compute_loss(outputs, fam_tgts, feat_tgts, epoch=epoch, feat_weights=feat_wts)
            total_loss += loss.item() * len(imgs)

            preds = outputs["family_logits"].argmax(1)
            correct_primary += (preds == fam_tgts).sum().item()
            total_samples   += len(imgs)

            # Feature accuracy
            for fam in ROCK_FAMILIES:
                fam_idx  = FAMILY_TO_IDX[fam]
                fam_mask = fam_tgts == fam_idx
                if not fam_mask.any():
                    continue
                for feat, tgt in feat_tgts[fam].items():
                    valid = fam_mask & (tgt != IGNORE_INDEX)
                    if not valid.any():
                        continue
                    logits  = outputs["feature_logits"][fam][feat][valid]
                    correct = (logits.argmax(1) == tgt[valid]).sum().item()
                    feat_correct[f"{fam}.{feat}"] += correct
                    feat_total[f"{fam}.{feat}"]   += valid.sum().item()

    avg_loss    = total_loss / max(total_samples, 1)
    primary_acc = correct_primary / max(total_samples, 1)
    feat_acc    = {k: feat_correct[k] / max(feat_total[k], 1) for k in feat_total}

    return {"loss": avg_loss, "primary_acc": primary_acc, "feat_acc": feat_acc}

print("evaluate() defined")

## 11. Training loop

In [ ]:
import time
from pathlib import Path

best_val_acc    = 0.0
patience_count  = 0
history         = []

WEIGHTS_OUT.parent.mkdir(parents=True, exist_ok=True)
CKPT_DIR = WEIGHTS_OUT.parent / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

for epoch in range(CFG["epochs"]):
    model.train()
    t0  = time.time()
    running = {"total": 0.0, "primary": 0.0, "secondary": 0.0}
    n_batches = 0

    for imgs, fam_tgts, feat_tgts, feat_wts in train_loader:
        imgs      = imgs.to(DEVICE)
        fam_tgts  = fam_tgts.to(DEVICE)
        feat_tgts = {fam: {f: t.to(DEVICE) for f, t in v.items()} for fam, v in feat_tgts.items()}
        feat_wts  = {fam: {f: w.to(DEVICE) for f, w in v.items()} for fam, v in feat_wts.items()}

        optimizer.zero_grad()
        outputs    = model(imgs)
        loss, comp = compute_loss(outputs, fam_tgts, feat_tgts, epoch=epoch,
                                  label_smoothing=CFG["label_smoothing"],
                                  feat_weights=feat_wts)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        optimizer.step()

        for k in running:
            running[k] += comp.get(k, 0.0)
        n_batches += 1

    scheduler.step()

    # Validation
    val_metrics = evaluate(model, val_loader, epoch=epoch)
    val_acc     = val_metrics["primary_acc"]

    elapsed = time.time() - t0
    train_loss = running["total"] / max(n_batches, 1)
    row = {
        "epoch":      epoch,
        "train_loss": round(train_loss, 4),
        "val_loss":   round(val_metrics["loss"], 4),
        "val_acc":    round(val_acc, 4),
        "secondary":  round(running["secondary"] / max(n_batches,1), 4),
        "lr":         round(scheduler.get_last_lr()[0], 6),
        "elapsed_s":  round(elapsed, 1),
    }
    history.append(row)

    flag = ""
    if val_acc > best_val_acc:
        best_val_acc   = val_acc
        patience_count = 0
        save_checkpoint(model, WEIGHTS_OUT, metadata={"epoch": epoch, "val_acc": val_acc})
        flag = " ← best"
    else:
        patience_count += 1

    if (epoch + 1) % CFG["save_every"] == 0:
        save_checkpoint(model, CKPT_DIR / f"epoch_{epoch:03d}.pt", metadata=row)

    print(f"Epoch {epoch:3d} | train={train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | "
          f"val_acc={val_acc:.3f} | lr={row['lr']:.2e} | {elapsed:.0f}s{flag}")

    if patience_count >= CFG["patience"]:
        print(f"Early stopping at epoch {epoch} (no improvement for {patience_count} epochs)")
        break

print(f"Best val acc: {best_val_acc:.4f}  →  {WEIGHTS_OUT}")

## 12. Feature accuracy (val)

In [ ]:
best_model, _ = load_checkpoint(WEIGHTS_OUT, DEVICE)

val_m = evaluate(best_model, val_loader)
print(f"Val primary acc: {val_m['primary_acc']:.4f}")
print("\nFeature accuracies (val):")
for k in sorted(val_m["feat_acc"]):
    print(f"  {k:<40} {val_m['feat_acc'][k]:.3f}")

## 13. Temperature calibration
Fits a separate temperature scalar per head on the validation set.

In [ ]:
import torch.optim as optim_

def calibrate_temperature(model, loader, n_epochs=50, lr=0.01):
    """Fit temperature scalars on val set. Freeze all other params."""
    model.eval()

    # Enable grad only on temperature params
    for p in model.parameters():
        p.requires_grad_(False)
    model.primary_temp.requires_grad_(True)
    for fam in ROCK_FAMILIES:
        for feat in model.feature_temps[fam]:
            model.feature_temps[fam][feat].requires_grad_(True)

    temp_params = (
        [model.primary_temp]
        + [model.feature_temps[fam][feat] for fam in ROCK_FAMILIES for feat in model.feature_temps[fam]]
    )
    cal_opt = optim_.LBFGS(temp_params, lr=lr, max_iter=50)

    # Collect all logits + targets first (val set is small)
    all_fam_logits  = []
    all_fam_tgts    = []
    all_feat_logits = {fam: {f: [] for f in FEATURE_SCHEMAS[fam]} for fam in ROCK_FAMILIES}
    all_feat_tgts   = {fam: {f: [] for f in FEATURE_SCHEMAS[fam]} for fam in ROCK_FAMILIES}

    with torch.no_grad():
        for imgs, fam_tgts, feat_tgts, feat_wts in loader:
            imgs      = imgs.to(DEVICE)
            fam_tgts  = fam_tgts.to(DEVICE)
            feat_tgts = {fam: {f: t.to(DEVICE) for f, t in v.items()} for fam, v in feat_tgts.items()}
            out       = model(imgs)
            all_fam_logits.append(out["family_logits"])
            all_fam_tgts.append(fam_tgts)
            for fam in ROCK_FAMILIES:
                for feat in FEATURE_SCHEMAS[fam]:
                    all_feat_logits[fam][feat].append(out["feature_logits"][fam][feat])
                    all_feat_tgts[fam][feat].append(feat_tgts[fam][feat])

    all_fam_logits = torch.cat(all_fam_logits)
    all_fam_tgts   = torch.cat(all_fam_tgts)
    for fam in ROCK_FAMILIES:
        for feat in FEATURE_SCHEMAS[fam]:
            all_feat_logits[fam][feat] = torch.cat(all_feat_logits[fam][feat])
            all_feat_tgts[fam][feat]   = torch.cat(all_feat_tgts[fam][feat])

    def nll():
        cal_opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(
            all_fam_logits / model.primary_temp.clamp(min=0.1), all_fam_tgts
        )
        for fam in ROCK_FAMILIES:
            fam_idx  = FAMILY_TO_IDX[fam]
            fam_mask = all_fam_tgts == fam_idx
            for feat in FEATURE_SCHEMAS[fam]:
                tgt   = all_feat_tgts[fam][feat]
                valid = fam_mask & (tgt != IGNORE_INDEX)
                if valid.any():
                    temp = model.feature_temps[fam][feat].clamp(min=0.1)
                    loss = loss + torch.nn.functional.cross_entropy(
                        all_feat_logits[fam][feat][valid] / temp, tgt[valid]
                    )
        loss.backward()
        return loss

    for _ in range(n_epochs):
        cal_opt.step(nll)

    print(f"Primary temp: {model.primary_temp.item():.4f}")
    for fam in ROCK_FAMILIES:
        for feat in model.feature_temps[fam]:
            print(f"  {fam}.{feat}: {model.feature_temps[fam][feat].item():.4f}")

    # Freeze temps again
    for p in model.parameters():
        p.requires_grad_(True)
    model.primary_temp.requires_grad_(False)
    for fam in ROCK_FAMILIES:
        for feat in model.feature_temps[fam]:
            model.feature_temps[fam][feat].requires_grad_(False)

    return model

best_model = calibrate_temperature(best_model, val_loader)
save_checkpoint(best_model, WEIGHTS_OUT, metadata={"calibrated": True})
print("Calibrated checkpoint saved.")

## 14. Final test evaluation

In [ ]:
test_loader = DataLoader(
    test_ds, batch_size=CFG["batch_size"],
    shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_feat_targets,
)

test_m = evaluate(best_model, test_loader)
print(f"Test primary accuracy: {test_m['primary_acc']:.4f}")
print("
Test feature accuracies:")
for k in sorted(test_m["feat_acc"]):
    print(f"  {k:<40} {test_m['feat_acc'][k]:.3f}")


## 14b. Confusion matrix & classification report

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.metrics import classification_report, confusion_matrix

best_model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for imgs, fam_tgts, _, __ in test_loader:
        imgs = imgs.to(DEVICE)
        out  = best_model(imgs)
        preds = out["family_logits"].argmax(1).cpu()
        all_preds.append(preds)
        all_targets.append(fam_tgts)

all_preds   = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()

# Only show rock classes (exclude 'other' if not present in test set)
present = sorted(set(all_targets))
labels  = [FAMILY_NAMES[i] for i in present]

# ── Classification report ─────────────────────────────────────────────
print(classification_report(all_targets, all_preds, labels=present, target_names=labels))

# ── Confusion matrix ──────────────────────────────────────────────────
cm      = confusion_matrix(all_targets, all_preds, labels=present)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, data, fmt, title in [
    (axes[0], cm,      "d",    "Counts"),
    (axes[1], cm_norm, ".2f",  "Row-normalised"),
]:
    im = ax.imshow(data, interpolation="nearest", cmap="Blues")
    fig.colorbar(im, ax=ax)
    ax.set(xticks=range(len(labels)), yticks=range(len(labels)),
           xticklabels=labels, yticklabels=labels,
           xlabel="Predicted", ylabel="True", title=f"Confusion matrix — {title}")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    thresh = data.max() / 2.0
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, format(data[i, j], fmt),
                    ha="center", va="center",
                    color="white" if data[i, j] > thresh else "black")

plt.tight_layout()
plt.savefig(str(WEIGHTS_OUT.parent / "confusion_matrix.png"), dpi=150)
plt.show()
print("Saved → confusion_matrix.png")


## 15. Training curves

In [ ]:
import matplotlib.pyplot as plt

epochs_  = [r["epoch"] for r in history]
tr_loss  = [r["train_loss"] for r in history]
va_loss  = [r["val_loss"] for r in history]
va_acc   = [r["val_acc"] for r in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_, tr_loss, label="train"); axes[0].plot(epochs_, va_loss, label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("epoch")
axes[1].plot(epochs_, va_acc); axes[1].set_title("Val primary accuracy"); axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.savefig(str(WEIGHTS_OUT.parent / "training_curves.png"), dpi=120)
plt.show()
